<a href="https://colab.research.google.com/github/Melhisk03/Music_Recommendation_System/blob/main/rec_sys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import scipy.sparse as sp
import os
from google.colab import drive

# --- 1. GOOGLE DRIVE BAĞLANTISI VE VERİ OKUMA ---

# Drive'ı bağla
drive.mount('/content/drive')

# Dosya yolunu hatadan aldığımız konuma göre güncelledik
file_path = '/content/Genel_Veriseti.xls'

# Dosya var mı kontrolü
if not os.path.exists(file_path):
    # Eğer o klasörde yoksa, belki ana dizindedir diye oraya da bakalım
    file_path_alt = '/content/drive/MyDrive/Genel_Veriseti.csv'
    if os.path.exists(file_path_alt):
        file_path = file_path_alt
    else:
        print(f"\nHATA: '{file_path}' konumunda dosya bulunamadı!")
        print("Lütfen dosyanızın Drive'daki tam yolunu kontrol edin.")
        df = pd.DataFrame() # Boş dataframe ile devam etmemesi için

if os.path.exists(file_path):
    print(f"\nDosya bulundu, okunuyor: {file_path}")

    # --- ROBUST (SAĞLAM) OKUMA YÖNTEMİ ---
    try:
        # Önce Python motoru ile ve ayırıcıyı otomatik algılayarak okumayı deneyelim.
        # on_bad_lines='skip' bozuk satırları atlar.
        df = pd.read_csv(file_path, sep=None, engine='python', on_bad_lines='skip')
        print("Otomatik ayırıcı algılama ile okundu.")
    except Exception as e:
        print(f"Otomatik okuma başarısız oldu, manuel deneniyor. Hata: {e}")
        try:
            # Olmazsa noktalı virgül deneyelim
            df = pd.read_csv(file_path, sep=";", on_bad_lines='skip')
            print("Noktalı virgül (;) ile okundu.")
        except:
            # O da olmazsa virgül deneyelim
            df = pd.read_csv(file_path, sep=",", on_bad_lines='skip')
            print("Virgül (,) ile okundu.")

    print(f"Toplam Satır: {len(df)}")
    print("Sütunlar:", df.columns.tolist())

# Eğer veri başarıyla okunduysa işlemlere devam et
if 'df' in locals() and not df.empty:
    # Gerekli Olan Özellikler (Features)
    features = [
        'danceability','energy','valence','acousticness','instrumentalness',
        'tempo','loudness','speechiness','liveness','popularity','release_year'
    ]

    # Sütun Eşleştirme
    cols_match = {}
    for f in features:
        found = False
        for c in df.columns:
            clean_col = c.lower().strip()
            if clean_col == f.lower():
                cols_match[f] = c
                found = True
                break
        if not found:
            # Eğer tam eşleşme yoksa, içinde geçiyor mu diye bakalım (örn: " spotify_popularity")
            for c in df.columns:
                if f.lower() in c.lower():
                    cols_match[f] = c
                    found = True
                    break

        if not found:
            print(f"UYARI: '{f}' sütunu verisetinde bulunamadı!")

    print("\nVeri temizleniyor ve sayıya dönüştürülüyor...")

    for feature_name, csv_col_name in cols_match.items():
        # Veri zaten sayısal gelmiş olabilir, önce string'e çevirip garantiye alalım
        # Virgülü nokta yap
        df[csv_col_name] = df[csv_col_name].astype(str).str.replace(',', '.', regex=False)
        # 'nan'-> NaN
        df[csv_col_name] = df[csv_col_name].replace('nan', np.nan)
        # Sayısal değere (float) dönüştür
        df[csv_col_name] = pd.to_numeric(df[csv_col_name], errors='coerce')

        if csv_col_name != feature_name:
            df.rename(columns={csv_col_name: feature_name}, inplace=True)

    # Genre kolonu bulma ve temizleme
    genre_col = None
    for c in df.columns:
        if 'genre' in c.lower():
            genre_col = c
            break

    if genre_col is None:
        print("UYARI: 'genre' sütunu bulunamadı, 'unknown' olarak oluşturuluyor.")
        df['genre'] = 'unknown'
        genre_col = 'genre'
    else:
        df[genre_col] = df[genre_col].fillna('unknown').astype(str)

    # --- 2. NUMERIC VE GENRE İŞLEME ---
    numeric_cols_final = [f for f in features if f in df.columns]
    print("Kullanılacak numeric kolonlar:", numeric_cols_final)

    df_numeric = df[numeric_cols_final].copy()
    df_numeric = df_numeric.fillna(df_numeric.mean())

    # Genre için TF-IDF
    tfidf = TfidfVectorizer(token_pattern=r"(?u)\b\w[\w+\s&-]*\b", lowercase=True)
    genre_tfidf = tfidf.fit_transform(df[genre_col])

    # Sayısal özellikleri ölçekleme
    scaler = StandardScaler()
    numeric_scaled = scaler.fit_transform(df_numeric)

    # Birleştirilmiş vektör (Sparse)
    numeric_sparse = sp.csr_matrix(numeric_scaled)
    combined = sp.hstack([numeric_sparse, genre_tfidf], format='csr')
    print("Combined vector shape:", combined.shape)

    # --- 3. MOOD AYARLARI ---
    mood_set = {
        'mutlu': {'valence': (0.6, 1.0), 'energy': (0.4, 1.0)},
        'üzgün': {'valence': (0.0, 0.4), 'energy': (0.0, 0.5)},
        'enerjik': {'valence': (0.3, 1.0), 'energy': (0.6, 1.0)},
        'rahat': {'valence': (0.3, 0.8), 'energy': (0.0, 0.5)},
        'nötr': {'valence': (0.4, 0.6), 'energy': (0.3, 0.6)},
        'tümü': {'valence': (0.0, 1.0), 'energy': (0.0, 1.0)},
    }

    # --- 4. ÖNERİ FONKSİYONU ---
    #def recommend_name(song_name=None, artist_name=None, top_n=10, genre_filter=None):


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Dosya bulundu, okunuyor: /content/Genel_Veriseti.xls
Otomatik ayırıcı algılama ile okundu.
Toplam Satır: 39388
Sütunlar: ['\ufeffid', 'name', 'artist', 'danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'tempo', 'loudness', 'speechiness', 'liveness', 'popularity', 'release_year', 'artist_popularity', 'genre']

Veri temizleniyor ve sayıya dönüştürülüyor...
Kullanılacak numeric kolonlar: ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'tempo', 'loudness', 'speechiness', 'liveness', 'popularity', 'release_year']
Combined vector shape: (39388, 324)


In [7]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import scipy.sparse as sp
import os
from google.colab import drive

# --- 1. GOOGLE DRIVE BAĞLANTISI VE VERİ OKUMA ---

# Drive'ı bağla
drive.mount('/content/drive')

# Dosya yolunu belirle
# Not: Dosyanızın yerini tam olarak buraya yazmalısınız.
file_path = '/content/drive/MyDrive/VeriSeti/Genel_Veriseti.csv'

# Dosya yolu kontrolü ve alternatif yollar
if not os.path.exists(file_path):
    # Alternatif: Ana dizini kontrol et
    file_path_alt = '/content/drive/MyDrive/Genel_Veriseti.csv'
    if os.path.exists(file_path_alt):
        file_path = file_path_alt
    else:
        print(f"\nHATA: '{file_path}' konumunda dosya bulunamadı!")
        print("Lütfen dosyanızın Drive'daki tam yolunu kontrol edin.")
        df = pd.DataFrame() # Hata almamak için boş df

# Dosya varsa okumaya çalış
if os.path.exists(file_path):
    print(f"\nDosya bulundu, okunuyor: {file_path}")

    # --- ROBUST (SAĞLAM) OKUMA YÖNTEMİ ---
    try:
        # 1. Yöntem: Python motoru ile otomatik ayırıcı algılama
        df = pd.read_csv(file_path, sep=None, engine='python', on_bad_lines='skip')
        print("Başarılı: Otomatik ayırıcı algılama ile okundu.")
    except Exception as e:
        print(f"Otomatik okuma başarısız, manuel deneniyor: {e}")
        try:
            # 2. Yöntem: Noktalı virgül (;)
            df = pd.read_csv(file_path, sep=";", on_bad_lines='skip')
            print("Başarılı: Noktalı virgül (;) ile okundu.")
        except:
            try:
                # 3. Yöntem: Virgül (,)
                df = pd.read_csv(file_path, sep=",", on_bad_lines='skip')
                print("Başarılı: Virgül (,) ile okundu.")
            except:
                print("HATA: Dosya hiçbir yöntemle okunamadı.")
                df = pd.DataFrame()

# Eğer veri başarıyla okunduysa işlemlere devam et
if 'df' in locals() and not df.empty:
    print(f"Toplam Satır: {len(df)}")

    # Gerekli Olan Özellikler (Features)
    features = [
        'danceability','energy','valence','acousticness','instrumentalness',
        'tempo','loudness','speechiness','liveness','popularity','release_year'
    ]

    # Sütun Eşleştirme (Büyük/küçük harf duyarlılığını kaldırma)
    cols_match = {}
    for f in features:
        found = False
        for c in df.columns:
            # Temiz sütun adı
            clean_col = str(c).lower().strip()
            if clean_col == f.lower():
                cols_match[f] = c
                found = True
                break

        # Tam eşleşme yoksa "içeren"e bak (örn: " spotify_popularity")
        if not found:
            for c in df.columns:
                if f.lower() in str(c).lower():
                    cols_match[f] = c
                    found = True
                    break

        if not found:
            print(f"UYARI: '{f}' sütunu verisetinde bulunamadı!")

    print("\nVeri temizleniyor ve sayıya dönüştürülüyor...")

    for feature_name, csv_col_name in cols_match.items():
        # Veri tiplerini düzelt
        df[csv_col_name] = df[csv_col_name].astype(str).str.replace(',', '.', regex=False)
        df[csv_col_name] = df[csv_col_name].replace('nan', np.nan)
        df[csv_col_name] = pd.to_numeric(df[csv_col_name], errors='coerce')

        # Sütun ismini standartlaştır
        if csv_col_name != feature_name:
            df.rename(columns={csv_col_name: feature_name}, inplace=True)

    # Genre (Tür) kolonu bulma
    genre_col = None
    for c in df.columns:
        if 'genre' in str(c).lower():
            genre_col = c
            break

    if genre_col is None:
        print("UYARI: 'genre' sütunu bulunamadı, 'unknown' olarak oluşturuluyor.")
        df['genre'] = 'unknown'
        genre_col = 'genre'
    else:
        df[genre_col] = df[genre_col].fillna('unknown').astype(str)

    # --- 2. NUMERIC VE GENRE İŞLEME ---
    numeric_cols_final = [f for f in features if f in df.columns]
    print("Kullanılacak numeric kolonlar:", numeric_cols_final)

    df_numeric = df[numeric_cols_final].copy()
    df_numeric = df_numeric.fillna(df_numeric.mean())

    # Genre için TF-IDF
    tfidf = TfidfVectorizer(token_pattern=r"(?u)\b\w[\w+\s&-]*\b", lowercase=True)
    genre_tfidf = tfidf.fit_transform(df[genre_col])

    # Sayısal özellikleri ölçekleme
    scaler = StandardScaler()
    numeric_scaled = scaler.fit_transform(df_numeric)

    # Birleştirilmiş vektör (Sparse Matrix)
    numeric_sparse = sp.csr_matrix(numeric_scaled)
    combined = sp.hstack([numeric_sparse, genre_tfidf], format='csr')
    print("Combined vector shape:", combined.shape)

    # --- 3. MOOD (MOD) AYARLARI ---
    mood_set = {
        'mutlu': {'valence': (0.6, 1.0), 'energy': (0.4, 1.0)},
        'üzgün': {'valence': (0.0, 0.4), 'energy': (0.0, 0.5)},
        'enerjik': {'valence': (0.3, 1.0), 'energy': (0.6, 1.0)},
        'rahat': {'valence': (0.3, 0.8), 'energy': (0.0, 0.5)},
        'nötr': {'valence': (0.4, 0.6), 'energy': (0.3, 0.6)},
        'tümü': {'valence': (0.0, 1.0), 'energy': (0.0, 1.0)},
    }

    # --- 4. ÖNERİ FONKSİYONU (COSINE SIMILARITY) ---
    def recommend_name(
        song_name=None,
        artist_name=None,
        top_n=10,
        genre_filter=None,
        mood='tümü'
    ):
        # Sorgu Vektörü Hazırlama
        match = pd.DataFrame()
        query_vec = None

        if song_name:
            if artist_name:
                # Hem şarkı hem sanatçı adı varsa
                match = df[
                    (df["name"].astype(str).str.lower() == song_name.lower()) &
                    (df["artist"].astype(str).str.lower() == artist_name.lower())
                ]
                if match.empty:
                    match = df[
                        df["name"].astype(str).str.lower().str.contains(song_name.lower(), na=False) &
                        df["artist"].astype(str).str.lower().str.contains(artist_name.lower(), na=False)
                    ]
            else:
                # Sadece şarkı adı varsa
                match = df[df["name"].astype(str).str.lower() == song_name.lower()]
                if match.empty:
                    match = df[df["name"].astype(str).str.lower().str.contains(song_name.lower(), na=False)]

            if match.empty:
                print(f"UYARI: Şarkı '{song_name}' bulunamadı.")
                return pd.DataFrame()

            idx = match.index[0]
            query_vec = combined[idx]

        elif artist_name:
            # Sadece sanatçı adı varsa -> Sanatçının şarkılarının ortalaması
            artist_songs = df[df["artist"].astype(str).str.lower() == artist_name.lower()]
            if artist_songs.empty:
                artist_songs = df[df["artist"].astype(str).str.lower().str.contains(artist_name.lower(), na=False)]

            if artist_songs.empty:
                print(f"UYARI: Artist '{artist_name}' bulunamadı.")
                return pd.DataFrame()

            query_vec = combined[artist_songs.index].mean(axis=0)
            query_vec = sp.csr_matrix(query_vec)
        else:
            return pd.DataFrame()

        # Mood Filtresi
        mood_key = mood if mood in mood_set else 'tümü'
        vmin, vmax = mood_set[mood_key]["valence"]
        emin, emax = mood_set[mood_key]["energy"]

        mood_mask = (
            (df["valence"] >= vmin) & (df["valence"] <= vmax) &
            (df["energy"] >= emin) & (df["energy"] <= emax)
        )
        adaylar = df[mood_mask].index

        # Genre Filtresi
        if genre_filter:
            genre_mask = df[genre_col].str.contains(genre_filter, case=False, na=False)
            adaylar = df[genre_mask & df.index.isin(adaylar)].index

        if len(adaylar) == 0:
            return pd.DataFrame()

        # Benzerlik Hesaplama
        cand_matrix = combined[adaylar]
        sims = cosine_similarity(query_vec, cand_matrix).flatten()
        sorted_idx = np.argsort(sims)[::-1]
        adaylar = np.array(adaylar)

        results = []
        for i in sorted_idx:
            song_i = adaylar[i]
            # Sorgulanan şarkının kendisini sonuçlarda gösterme
            if song_name and 'idx' in locals() and song_i == idx:
                continue

            results.append({
                "name": df.loc[song_i, "name"],
                "artist": df.loc[song_i, "artist"],
                "genre": df.loc[song_i, genre_col],
                "similarity": round(float(sims[i]), 4)
            })
            if len(results) >= top_n:
                break
        return pd.DataFrame(results)

    # --- 5. PCA İŞLEMLERİ ---
    print("\n--- PCA Modeli Hazırlanıyor ---")

    df_numeric_pca = df[numeric_cols_final].fillna(df[numeric_cols_final].mean())
    scaler_pca = StandardScaler()
    numeric_scaled_pca = scaler_pca.fit_transform(df_numeric_pca)

    pca = PCA(n_components=0.95)
    pca_components = pca.fit_transform(numeric_scaled_pca)
    print("PCA Output shape:", pca_components.shape)

    def recommend_pca(
        song_name=None,
        artist_name=None,
        top_n=10,
        genre_filter=None,
        mood='tümü'
    ):
        match = pd.DataFrame()
        query_vec = None

        if song_name:
            if artist_name:
                match = df[
                    (df["name"].astype(str).str.lower() == song_name.lower()) &
                    (df["artist"].astype(str).str.lower() == artist_name.lower())
                ]
                if match.empty:
                    match = df[
                        df["name"].astype(str).str.lower().str.contains(song_name.lower(), na=False) &
                        df["artist"].astype(str).str.lower().str.contains(artist_name.lower(), na=False)
                    ]
            else:
                match = df[df["name"].astype(str).str.lower() == song_name.lower()]
                if match.empty:
                    match = df[df["name"].astype(str).str.lower().str.contains(song_name.lower(), na=False)]

            if match.empty:
                print(f"UYARI: Şarkı '{song_name}' bulunamadı.")
                return pd.DataFrame()

            idx = match.index[0]
            query_vec = pca_components[idx].reshape(1, -1)

        elif artist_name:
            artist_songs = df[df["artist"].astype(str).str.lower() == artist_name.lower()]
            if artist_songs.empty:
                artist_songs = df[df["artist"].astype(str).str.lower().str.contains(artist_name.lower(), na=False)]

            if artist_songs.empty:
                print(f"UYARI: Artist '{artist_name}' bulunamadı.")
                return pd.DataFrame()
            query_vec = pca_components[artist_songs.index].mean(axis=0)
            query_vec = query_vec.reshape(1, -1)
        else:
            return pd.DataFrame()

        mood_key = mood if mood in mood_set else 'tümü'
        vmin, vmax = mood_set[mood_key]['valence']
        emin, emax = mood_set[mood_key]['energy']

        mood_mask = (
            (df["valence"] >= vmin) & (df["valence"] <= vmax) &
            (df["energy"] >= emin) & (df["energy"] <= emax)
        )
        candidate_idx = df[mood_mask].index

        if genre_filter:
            gmask = df[genre_col].str.contains(genre_filter, case=False, na=False)
            candidate_idx = df[gmask & df.index.isin(candidate_idx)].index

        if len(candidate_idx) == 0:
            return pd.DataFrame()

        candidate_matrix = pca_components[candidate_idx]
        sims = cosine_similarity(query_vec, candidate_matrix).flatten()
        sorted_idx = np.argsort(sims)[::-1]
        candidate_idx = np.array(candidate_idx)

        results = []
        for i in sorted_idx:
            song_i = candidate_idx[i]
            if song_name and 'idx' in locals() and song_i == idx:
                continue

            results.append({
                "name": df.loc[song_i, "name"],
                "artist": df.loc[song_i, "artist"],
                "genre": df.loc[song_i, genre_col],
                "similarity": round(float(sims[i]), 4)
            })
            if len(results) >= top_n:
                break

        return pd.DataFrame(results)

    # --- 6. TESTLER ---
    print("\n--- TESTLER ÇALIŞTIRILIYOR ---")
    try:
        print("\n1. Cosine Similarity (Way Down We Go):")
        print(recommend_name(artist_name="Metallica"))

        print("\n2. Cosine Similarity (Rammstein - Sonne):")
        print(recommend_name(artist_name="Rammstein", song_name="Sonne"))

        print("\n3. PCA (Dua Lipa):")
        print(recommend_pca(artist_name="Dua Lipa"))
    except Exception as e:
        print("Test sırasında hata oluştu:", e)

else:
    print("Veri yüklenemediği için işlemler iptal edildi.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Dosya bulundu, okunuyor: /content/drive/MyDrive/Genel_Veriseti.csv
Başarılı: Otomatik ayırıcı algılama ile okundu.
Toplam Satır: 27373

Veri temizleniyor ve sayıya dönüştürülüyor...
Kullanılacak numeric kolonlar: ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'tempo', 'loudness', 'speechiness', 'liveness', 'popularity', 'release_year']
Combined vector shape: (27373, 118)

--- PCA Modeli Hazırlanıyor ---
PCA Output shape: (27373, 10)

--- TESTLER ÇALIŞTIRILIYOR ---

1. Cosine Similarity (Way Down We Go):
                               name                 artist   genre  similarity
0                Seven - Remastered  Sunny Day Real Estate  rock;;      0.9318
1         Rejoice - Remastered 2008                     U2  rock;;      0.9237
2  Like A Song... - Remastered 2008                     U2  rock;;      0.9192
3                 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import scipy.sparse as sp
import os
from google.colab import drive
import sys

# --- 1. GOOGLE DRIVE BAĞLANTISI VE VERİ OKUMA ---

# Drive'ı bağla
drive.mount('/content/drive')

# Dosya yolunu belirle
# Dosyanızın tam yolunu buraya yazdığınızdan emin olun.
file_path = '/content/drive/MyDrive/VeriSeti/Genel_Veriseti.csv'

# Dosya yolu kontrolü ve alternatif yollar
if not os.path.exists(file_path):
    file_path_alt = '/content/drive/MyDrive/Genel_Veriseti.csv'
    if os.path.exists(file_path_alt):
        file_path = file_path_alt
    else:
        print(f"\nHATA: '{file_path}' konumunda dosya bulunamadı!")
        print("Lütfen dosyanızın Drive'daki tam yolunu kontrol edin.")
        df = pd.DataFrame()

# Dosya varsa okumaya çalış
if os.path.exists(file_path):
    print(f"\nDosya bulundu, okunuyor: {file_path}")
    try:
        df = pd.read_csv(file_path, sep=None, engine='python', on_bad_lines='skip')
        print("Başarılı: Otomatik ayırıcı algılama ile okundu.")
    except Exception as e:
        print(f"Otomatik okuma başarısız, manuel deneniyor: {e}")
        try:
            df = pd.read_csv(file_path, sep=";", on_bad_lines='skip')
            print("Başarılı: Noktalı virgül (;) ile okundu.")
        except:
            try:
                df = pd.read_csv(file_path, sep=",", on_bad_lines='skip')
                print("Başarılı: Virgül (,) ile okundu.")
            except:
                print("HATA: Dosya hiçbir yöntemle okunamadı.")
                df = pd.DataFrame()

# Eğer veri başarıyla okunduysa işlemlere devam et
if 'df' in locals() and not df.empty:
    print(f"Toplam Satır: {len(df)}")

    # Gerekli Olan Özellikler
    features = [
        'danceability','energy','valence','acousticness','instrumentalness',
        'tempo','loudness','speechiness','liveness','popularity','release_year'
    ]

    # Sütun Eşleştirme
    cols_match = {}
    for f in features:
        found = False
        for c in df.columns:
            clean_col = str(c).lower().strip()
            if clean_col == f.lower():
                cols_match[f] = c
                found = True
                break
        if not found:
            for c in df.columns:
                if f.lower() in str(c).lower():
                    cols_match[f] = c
                    found = True
                    break
        if not found:
            print(f"UYARI: '{f}' sütunu verisetinde bulunamadı!")

    print("\nVeri temizleniyor ve sayıya dönüştürülüyor...")

    for feature_name, csv_col_name in cols_match.items():
        df[csv_col_name] = df[csv_col_name].astype(str).str.replace(',', '.', regex=False)
        df[csv_col_name] = df[csv_col_name].replace('nan', np.nan)
        df[csv_col_name] = pd.to_numeric(df[csv_col_name], errors='coerce')

        if csv_col_name != feature_name:
            df.rename(columns={csv_col_name: feature_name}, inplace=True)

    # Genre (Tür) kolonu
    genre_col = None
    for c in df.columns:
        if 'genre' in str(c).lower():
            genre_col = c
            break

    if genre_col is None:
        df['genre'] = 'unknown'
        genre_col = 'genre'
    else:
        df[genre_col] = df[genre_col].fillna('unknown').astype(str)

    # --- 2. NUMERIC VE GENRE İŞLEME ---
    numeric_cols_final = [f for f in features if f in df.columns]
    df_numeric = df[numeric_cols_final].copy()
    df_numeric = df_numeric.fillna(df_numeric.mean())

    tfidf = TfidfVectorizer(token_pattern=r"(?u)\b\w[\w+\s&-]*\b", lowercase=True)
    genre_tfidf = tfidf.fit_transform(df[genre_col])

    scaler = StandardScaler()
    numeric_scaled = scaler.fit_transform(df_numeric)

    numeric_sparse = sp.csr_matrix(numeric_scaled)
    combined = sp.hstack([numeric_sparse, genre_tfidf], format='csr')

    # --- 3. MOOD AYARLARI ---
    mood_set = {
        'mutlu': {'valence': (0.6, 1.0), 'energy': (0.4, 1.0)},
        'üzgün': {'valence': (0.0, 0.4), 'energy': (0.0, 0.5)},
        'enerjik': {'valence': (0.3, 1.0), 'energy': (0.6, 1.0)},
        'rahat': {'valence': (0.3, 0.8), 'energy': (0.0, 0.5)},
        'nötr': {'valence': (0.4, 0.6), 'energy': (0.3, 0.6)},
        'tümü': {'valence': (0.0, 1.0), 'energy': (0.0, 1.0)},
    }

    # --- 4. ÖNERİ FONKSİYONU (COSINE) ---
    def recommend_name(song_name=None, artist_name=None, top_n=10, genre_filter=None, mood='tümü'):
        match = pd.DataFrame()
        query_vec = None

        if song_name:
            if artist_name:
                match = df[
                    (df["name"].astype(str).str.lower() == song_name.lower()) &
                    (df["artist"].astype(str).str.lower() == artist_name.lower())
                ]
                if match.empty:
                    match = df[
                        df["name"].astype(str).str.lower().str.contains(song_name.lower(), na=False) &
                        df["artist"].astype(str).str.lower().str.contains(artist_name.lower(), na=False)
                    ]
            else:
                match = df[df["name"].astype(str).str.lower() == song_name.lower()]
                if match.empty:
                    match = df[df["name"].astype(str).str.lower().str.contains(song_name.lower(), na=False)]

            if match.empty:
                print(f"UYARI: Şarkı '{song_name}' bulunamadı.")
                return pd.DataFrame()

            idx = match.index[0]
            query_vec = combined[idx]

        elif artist_name:
            artist_songs = df[df["artist"].astype(str).str.lower() == artist_name.lower()]
            if artist_songs.empty:
                artist_songs = df[df["artist"].astype(str).str.lower().str.contains(artist_name.lower(), na=False)]

            if artist_songs.empty:
                print(f"UYARI: Artist '{artist_name}' bulunamadı.")
                return pd.DataFrame()

            query_vec = combined[artist_songs.index].mean(axis=0)
            query_vec = sp.csr_matrix(query_vec)
        else:
            return pd.DataFrame()

        mood_key = mood if mood in mood_set else 'tümü'
        vmin, vmax = mood_set[mood_key]["valence"]
        emin, emax = mood_set[mood_key]["energy"]

        mood_mask = (
            (df["valence"] >= vmin) & (df["valence"] <= vmax) &
            (df["energy"] >= emin) & (df["energy"] <= emax)
        )
        adaylar = df[mood_mask].index

        if genre_filter:
            genre_mask = df[genre_col].str.contains(genre_filter, case=False, na=False)
            adaylar = df[genre_mask & df.index.isin(adaylar)].index

        if len(adaylar) == 0:
            return pd.DataFrame()

        cand_matrix = combined[adaylar]
        sims = cosine_similarity(query_vec, cand_matrix).flatten()
        sorted_idx = np.argsort(sims)[::-1]
        adaylar = np.array(adaylar)

        results = []
        for i in sorted_idx:
            song_i = adaylar[i]
            if song_name and 'idx' in locals() and song_i == idx:
                continue

            results.append({
                "name": df.loc[song_i, "name"],
                "artist": df.loc[song_i, "artist"],
                "genre": df.loc[song_i, genre_col],
                "valence": df.loc[song_i, "valence"],
                "energy": df.loc[song_i, "energy"],
                "similarity": round(float(sims[i]), 4)
            })
            if len(results) >= top_n:
                break
        return pd.DataFrame(results)

    # --- 5. PCA İŞLEMLERİ ---
    print("\n--- PCA Modeli Hazırlanıyor ---")
    df_numeric_pca = df[numeric_cols_final].fillna(df[numeric_cols_final].mean())
    scaler_pca = StandardScaler()
    numeric_scaled_pca = scaler_pca.fit_transform(df_numeric_pca)

    pca = PCA(n_components=0.95)
    pca_components = pca.fit_transform(numeric_scaled_pca)

    def recommend_pca(song_name=None, artist_name=None, top_n=10, genre_filter=None, mood='tümü'):
        match = pd.DataFrame()
        query_vec = None

        if song_name:
            if artist_name:
                match = df[
                    (df["name"].astype(str).str.lower() == song_name.lower()) &
                    (df["artist"].astype(str).str.lower() == artist_name.lower())
                ]
                if match.empty:
                    match = df[
                        df["name"].astype(str).str.lower().str.contains(song_name.lower(), na=False) &
                        df["artist"].astype(str).str.lower().str.contains(artist_name.lower(), na=False)
                    ]
            else:
                match = df[df["name"].astype(str).str.lower() == song_name.lower()]
                if match.empty:
                    match = df[df["name"].astype(str).str.lower().str.contains(song_name.lower(), na=False)]

            if match.empty:
                print(f"UYARI: Şarkı '{song_name}' bulunamadı.")
                return pd.DataFrame()

            idx = match.index[0]
            query_vec = pca_components[idx].reshape(1, -1)

        elif artist_name:
            artist_songs = df[df["artist"].astype(str).str.lower() == artist_name.lower()]
            if artist_songs.empty:
                artist_songs = df[df["artist"].astype(str).str.lower().str.contains(artist_name.lower(), na=False)]

            if artist_songs.empty:
                print(f"UYARI: Artist '{artist_name}' bulunamadı.")
                return pd.DataFrame()
            query_vec = pca_components[artist_songs.index].mean(axis=0)
            query_vec = query_vec.reshape(1, -1)
        else:
            return pd.DataFrame()

        mood_key = mood if mood in mood_set else 'tümü'
        vmin, vmax = mood_set[mood_key]['valence']
        emin, emax = mood_set[mood_key]['energy']

        mood_mask = (
            (df["valence"] >= vmin) & (df["valence"] <= vmax) &
            (df["energy"] >= emin) & (df["energy"] <= emax)
        )
        candidate_idx = df[mood_mask].index

        if genre_filter:
            gmask = df[genre_col].str.contains(genre_filter, case=False, na=False)
            candidate_idx = df[gmask & df.index.isin(candidate_idx)].index

        if len(candidate_idx) == 0:
            return pd.DataFrame()

        candidate_matrix = pca_components[candidate_idx]
        sims = cosine_similarity(query_vec, candidate_matrix).flatten()
        sorted_idx = np.argsort(sims)[::-1]
        candidate_idx = np.array(candidate_idx)

        results = []
        for i in sorted_idx:
            song_i = candidate_idx[i]
            if song_name and 'idx' in locals() and song_i == idx:
                continue

            # BURADA HATA OLUŞUYORDU, DÜZELTİLDİ:
            results.append({
                "name": df.loc[song_i, "name"],
                "artist": df.loc[song_i, "artist"],
                "genre": df.loc[song_i, genre_col],
                "valence": df.loc[song_i, "valence"],
                "energy": df.loc[song_i, "energy"],
                "similarity": round(float(sims[i]), 4)
            })
            if len(results) >= top_n:
                break

        return pd.DataFrame(results)

    # --- 6. INTERAKTIF ARAYÜZ ---
    def interaktif_sorgu():
        print("="*50)
        print("🎵 MÜZİK ÖNERİ ASİSTANI BAŞLATILDI 🎵")
        print("Çıkmak için 'q' veya 'exit' yazabilirsiniz.")
        print("="*50)

        while True:
            print("\nNE YAPMAK İSTERSİNİZ?")
            print("1. Bir Sanatçıya göre öneri al (Örn: Metallica sevenler ne dinler?)")
            print("2. Belirli bir Şarkıya göre öneri al (Örn: 'Sonne' şarkısına benzerler)")
            print("q. Çıkış")

            secim = input("\nSeçiminiz (1/2/q): ").strip().lower()

            if secim in ['q', 'exit', 'cikis']:
                print("👋 Görüşmek üzere!")
                break

            if secim not in ['1', '2']:
                print("❌ Geçersiz seçim! Lütfen 1 veya 2'yi tuşlayın.")
                continue

            try:
                artist_input = input("Sanatçı Adı Girin (Örn: Rammstein): ").strip()
                if not artist_input:
                    print("⚠️ Sanatçı adı boş olamaz!")
                    continue

                song_input = None
                if secim == '2':
                    song_input = input(f"{artist_input} - Şarkı Adı Girin (Örn: Sonne): ").strip()
                    if not song_input:
                        print("⚠️ Şarkı adı boş olamaz!")
                        continue

                print("\n--- Filtreler (Boş bırakıp ENTER'a basarsanız tümü seçilir) ---")
                print("Mood Seçenekleri: mutlu, üzgün, enerjik, rahat, nötr")
                mood_input = input("Mood (Mod) seçin: ").strip().lower()
                if not mood_input: mood_input = 'tümü'

                genre_input = input("Tür (Genre) filtresi (Örn: rock, pop, metal): ").strip().lower()
                if not genre_input: genre_input = None

                algo_secim = input("Algoritma? (1: Standart/Cosine, 2: PCA) [Varsayılan: 1]: ").strip()

                print(f"\n🔎 {artist_input} {'- ' + song_input if song_input else ''} için öneriler aranıyor...")

                results = pd.DataFrame()

                if algo_secim == '2':
                    results = recommend_pca(song_name=song_input, artist_name=artist_input, mood=mood_input, genre_filter=genre_input, top_n=10)
                    print("🧠 Model: PCA")
                else:
                    results = recommend_name(song_name=song_input, artist_name=artist_input, mood=mood_input, genre_filter=genre_input, top_n=10)
                    print("🧠 Model: Cosine Similarity")

                if not results.empty:
                    print("\n" + "-"*20 + " ÖNERİLER " + "-"*20)
                    display_cols = ['name', 'artist', 'genre', 'similarity']
                    # Sadece mood seçili değilse de ekleyelim ki kullanıcı görsün
                    display_cols.extend(['valence', 'energy'])

                    print(results[display_cols].to_string(index=False))
                    print("-" * 50)
                else:
                    print("\n❌ Kriterlere uygun sonuç bulunamadı veya şarkı verisetinde yok.")

            except Exception as e:
                print(f"\n❌ Bir hata oluştu: {e}")
                import traceback
                traceback.print_exc()
                print("Lütfen tekrar deneyin.")

    # Arayüzü başlat
    interaktif_sorgu()

else:
    print("Veri yüklenemediği için işlemler iptal edildi.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Dosya bulundu, okunuyor: /content/drive/MyDrive/VeriSeti/Genel_Veriseti.csv
Başarılı: Otomatik ayırıcı algılama ile okundu.
Toplam Satır: 39388

Veri temizleniyor ve sayıya dönüştürülüyor...

--- PCA Modeli Hazırlanıyor ---
🎵 MÜZİK ÖNERİ ASİSTANI BAŞLATILDI 🎵
Çıkmak için 'q' veya 'exit' yazabilirsiniz.

NE YAPMAK İSTERSİNİZ?
1. Bir Sanatçıya göre öneri al (Örn: Metallica sevenler ne dinler?)
2. Belirli bir Şarkıya göre öneri al (Örn: 'Sonne' şarkısına benzerler)
q. Çıkış

--- Filtreler (Boş bırakıp ENTER'a basarsanız tümü seçilir) ---
Mood Seçenekleri: mutlu, üzgün, enerjik, rahat, nötr

🔎 rammstein  için öneriler aranıyor...
🧠 Model: PCA

-------------------- ÖNERİLER --------------------
                    name       artist                                                    genre  similarity  valence  energy
                New Skin      Incubus nu metal,